# Fine-tuning BGE-M3 on the Telugu Bhagavad Gita

Contrastive fine-tuning of a bi-encoder so that **Telugu questions** land near
the **verse that answers them**. We are not teaching the model Telugu - it has
Telugu. We are reshaping the space for a retrieval task.

**Runtime → Change runtime type → T4 GPU** before running anything.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv


In [ ]:
%pip install -q -U sentence-transformers datasets accelerate


## 1. Upload data

From your machine, upload these three files out of `~/telugu-rag/finetune/`
plus the corpus:

- `train_triplets.jsonl` - (query, correct verse, hard negative)
- `held_queries.jsonl` - eval queries, generated ONLY from held-out verses
- `gita.jsonl` - the full 699-verse corpus (the retrieval haystack)


In [ ]:
from google.colab import files
import zipfile, os

up = files.upload()          # choose gita_finetune_data.zip

for name in up:
    if name.endswith('.zip'):
        zipfile.ZipFile(name).extractall('.')
        print('unzipped', name)

for f in ['train_triplets.jsonl', 'held_queries.jsonl', 'gita.jsonl']:
    print(f, '->', 'OK' if os.path.exists(f) else 'MISSING')


## 2. Load and sanity-check

The contamination check is not optional. If training queries and eval queries
come from the same verses, the eval measures memorisation and will show a large
fake improvement.


In [ ]:
import json

def jl(p): return [json.loads(l) for l in open(p, encoding='utf-8') if l.strip()]

triplets = jl('train_triplets.jsonl')
held_q   = jl('held_queries.jsonl')
corpus   = jl('gita.jsonl')

train_pos = {t['positive_id'] for t in triplets}
held_pos  = {q['verse_id'] for q in held_q}
overlap   = train_pos & held_pos

print(f'triplets        : {len(triplets)}')
print(f'held-out queries: {len(held_q)}')
print(f'corpus verses   : {len(corpus)}')
print(f'CONTAMINATION   : {len(overlap)} shared verses  <-- must be 0')
assert not overlap, 'train and eval share verses - fix the split before training'


## 3. Build the evaluator

`InformationRetrievalEvaluator` gives recall@k / NDCG / MRR. The haystack is the
**full 699-verse corpus**, so the task stays realistic - the model must find the
right verse among all of them, not just among held-out ones.


In [ ]:
import inspect
from sentence_transformers.evaluation import InformationRetrievalEvaluator

queries       = {f'q{i}': q['query'] for i, q in enumerate(held_q)}
relevant_docs = {f'q{i}': {q['verse_id']} for i, q in enumerate(held_q)}
corpus_map    = {r['id']: r['tatparyam_te'] for r in corpus}

# Recall comes from `precision_recall_at_k`, not `recall_at_k`, and the
# accepted kwargs shift between sentence-transformers releases. Pass only
# what this installed version actually declares.
wanted = dict(
    queries=queries, corpus=corpus_map, relevant_docs=relevant_docs,
    name='gita-held',
    ndcg_at_k=[10], mrr_at_k=[10], accuracy_at_k=[1, 5, 10],
    precision_recall_at_k=[1, 5, 10], map_at_k=[10],
    show_progress_bar=True, batch_size=32,
)
accepted = set(inspect.signature(InformationRetrievalEvaluator.__init__).parameters)
dropped  = sorted(set(wanted) - accepted)
if dropped: print('note: this version does not accept', dropped)
ir_eval = InformationRetrievalEvaluator(**{k: v for k, v in wanted.items() if k in accepted})
print(len(queries), 'eval queries over', len(corpus_map), 'verses')


## 4. Baseline - measure BEFORE training

Without this number the post-training number means nothing.


In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('BAAI/bge-m3')
model.max_seq_length = 512

baseline = ir_eval(model)
for k, v in sorted(baseline.items()):
    print(f'{k:<48} {v:.4f}')

# Derive the checkpoint-selection metric from the keys this version actually
# emits - the naming differs across sentence-transformers releases, and a
# wrong metric_for_best_model only fails partway through training.
cands = [k for k in baseline if k.endswith('recall@10')]
RECALL_KEY = next((k for k in cands if 'cosine' in k), cands[0])
BEST_METRIC = 'eval_' + RECALL_KEY
print('\nselecting checkpoints on:', BEST_METRIC)


## 5. Train

`CachedMultipleNegativesRankingLoss` is what lets a free T4 train at an effective
batch of 64-128. Batch size is the key quality lever here: it sets how many wrong
answers each query is contrasted against. `mini_batch_size` is the memory knob -
lower it if you OOM; it does not change the effective batch.

`learning_rate=2e-5` and 3 epochs are deliberately conservative. With a few
thousand examples it is easy to overfit into a model that knows this corpus and
has forgotten general Telugu.


In [ ]:
from datasets import Dataset
from sentence_transformers import SentenceTransformerTrainer, SentenceTransformerTrainingArguments
from sentence_transformers.training_args import BatchSamplers
from sentence_transformers.losses import CachedMultipleNegativesRankingLoss

train_ds = Dataset.from_dict({
    'anchor':   [t['anchor']   for t in triplets],
    'positive': [t['positive'] for t in triplets],
    'negative': [t['negative'] for t in triplets],
})
print(train_ds)

loss = CachedMultipleNegativesRankingLoss(model, mini_batch_size=8)

args = SentenceTransformerTrainingArguments(
    output_dir='bge-m3-gita',
    num_train_epochs=3,
    per_device_train_batch_size=64,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    fp16=True,
    # gradient_checkpointing is OFF on purpose. GradCache already keeps only
    # `mini_batch_size` items in memory, so it buys little here - and some
    # transformers versions pass `every_n_layers` to a model that rejects it
    # (TypeError: unexpected keyword argument 'every_n_layers').
    # If you OOM, lower mini_batch_size instead.
    eval_strategy='steps', eval_steps=25,
    save_strategy='steps', save_steps=25,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model=BEST_METRIC,
    logging_steps=10,
    # Each verse is the positive for 10 rows (5 queries x 2 negatives). Without
    # this sampler ~3 rows per batch of 64 share a positive, and the loss pushes
    # a query away from a document that is actually its correct answer.
    batch_sampler=BatchSamplers.NO_DUPLICATES,
    report_to='none',
)

trainer = SentenceTransformerTrainer(
    model=model, args=args, train_dataset=train_ds,
    loss=loss, evaluator=ir_eval,
)
trainer.train()


## 6. Compare


In [ ]:
after = ir_eval(model)
print(f"{'metric':<48}{'before':>10}{'after':>10}{'delta':>10}")
for k in sorted(baseline):
    b, a = baseline[k], after.get(k, 0)
    print(f'{k:<48}{b:>10.4f}{a:>10.4f}{a-b:>+10.4f}')


## 7. Save and download

Back on your Mac, point `MODEL_NAME` in `app/retrieval.py` at the unzipped
folder, rebuild the index, and re-run `app/evaluate.py` against your 14
hand-written questions - the honest test set that never touched training.


In [ ]:
model.save_pretrained('bge-m3-gita-final')

# The model is ~2.3 GB. A browser download that size frequently stalls, so
# save to Drive first and pull it down from there.
USE_DRIVE = True

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    !mkdir -p '/content/drive/MyDrive/telugu-rag'
    !cp -r bge-m3-gita-final '/content/drive/MyDrive/telugu-rag/'
    !du -sh '/content/drive/MyDrive/telugu-rag/bge-m3-gita-final'
    print('\nSaved to Drive > MyDrive > telugu-rag > bge-m3-gita-final')
else:
    !zip -qr bge-m3-gita-final.zip bge-m3-gita-final
    files.download('bge-m3-gita-final.zip')
